# Transcripción de VODs con Whisper v3 (distil-large-v3)

**Antes de empezar:** `Runtime > Change runtime type > T4 GPU`

| Celda | Descripción |
|-------|-------------|
| 1 | Instalar dependencias |
| 2 | Verificar GPU |
| 3 | **Configuración** ← editar aquí |
| 4 | Código core |
| 5 | Cargar modelo |
| 6 | Procesar **un VOD** |
| 7 | Procesar **lista de VODs** |
| 8 | Procesar **VODs pendientes** (sin transcripción, solo streamers configurados) |
| 9 | **Auto-loop** — procesa pendientes hasta completar todo |

**Mejoras v3:** modelo `distil-large-v3` (calidad de large-v3, velocidad de turbo), `condition_on_previous_text=False` (anti-hallucination), auto-fetch de VODs pendientes desde la API.

In [ ]:
# Celda 1 — Instalar dependencias
!pip install faster-whisper yt-dlp tqdm requests -q

In [ ]:
# Celda 2 — Verificar GPU
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'✓ GPU: {name}  ({vram:.1f} GB VRAM)')
else:
    print('❌ No GPU — ve a Runtime > Change runtime type > T4 GPU')

In [ ]:
# Celda 3 — Configuración (editar aquí)
VOD_ID     = "2710616501"   # ← VOD individual (celda 6)
MODEL_NAME = "Systran/faster-distil-whisper-large-v3"  # distil-large-v3: best speed/quality balance
API_BASE   = "https://backend.permisossubtel.cl/api"

# Streamers to process (celda 8 & 9)
STREAMER_LOGINS = ["leonblack", "shigity"]

# Lista de VODs para procesar en lote (celda 7)
VOD_LIST = [
    "2713338067", "2712286306", "2711483017", "2709567587", "2708619083",
    "2707914852", "2703589522", "2702613905", "2701711974", "2700993584",
    "2700908130", "2700093178", "2699120815", "2694743630", "2694651267",
    "2693926342", "2692238064", "2692060724", "2691020817", "2688400303",
    "2687477613", "2686538697", "2683617383", "2676733940", "2675035019",
    "2672370057", "2669529831", "2667773620", "2665878053", "2663272385",
    "2662149389",
]

In [ ]:
# Celda 4 — Código core
import gc
import json
import subprocess
import tempfile
import shutil
import time
import requests
from pathlib import Path
from tqdm import tqdm

UPLOAD_URL = f"{API_BASE}/transcripts/upload/"
VIDEOS_URL = f"{API_BASE}/videos/"
CACHE_DIR  = Path("/content/transcripts_cache")

WHISPER_SPEED_CPU = {
    "tiny": 80.0, "tiny.en": 80.0,
    "base": 40.0, "base.en": 40.0,
    "small": 15.0, "small.en": 15.0,
    "medium": 5.0, "medium.en": 5.0,
    "large-v1": 2.5, "large-v2": 2.5, "large-v3": 2.5, "large": 2.5,
    "large-v3-turbo": 20.0, "turbo": 20.0,
    "Systran/faster-distil-whisper-large-v3": 15.0,
}
WHISPER_SPEED_GPU = {
    "tiny": 200.0, "tiny.en": 200.0,
    "base": 150.0, "base.en": 150.0,
    "small": 80.0, "small.en": 80.0,
    "medium": 35.0, "medium.en": 35.0,
    "large-v1": 15.0, "large-v2": 15.0, "large-v3": 15.0, "large": 15.0,
    "large-v3-turbo": 70.0, "turbo": 70.0,
    "Systran/faster-distil-whisper-large-v3": 60.0,
}
DOWNLOAD_SPEED_X = 100.0


def fmt_seconds(s: float) -> str:
    s = int(s)
    h, rem = divmod(s, 3600)
    m, sec = divmod(rem, 60)
    parts = []
    if h: parts.append(f"{h}h")
    if m: parts.append(f"{m}m")
    parts.append(f"{sec}s")
    return " ".join(parts)


def get_video_info(vod_id: str) -> dict | None:
    try:
        resp = requests.get(f"{VIDEOS_URL}{vod_id}/", timeout=15)
        if resp.status_code == 200:
            return resp.json()
    except Exception:
        pass
    return None


def get_all_vods() -> list:
    vods, url = [], VIDEOS_URL
    while url:
        try:
            resp = requests.get(url, timeout=30)
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f"[ERROR] No se pudo obtener VODs: {e}")
            return vods
        if isinstance(data, list):
            vods.extend(data)
            break
        vods.extend(data.get("results", []))
        url = data.get("next")
    return vods


def get_pending_vods(streamer_logins: list) -> list:
    """Fetch VODs without transcripts for the given streamer logins."""
    all_vods = []
    for login in streamer_logins:
        url = f"{VIDEOS_URL}?has_transcript=false&streamer_login={login}&page_size=500"
        try:
            resp = requests.get(url, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            vods = data if isinstance(data, list) else data.get("results", [])
            all_vods.extend(vods)
            print(f"  {login}: {len(vods)} pending")
        except Exception as e:
            print(f"  {login}: ERROR — {e}")
    return all_vods


def download_audio(vod_id: str, dest: Path, duration_s) -> bool:
    twitch_url = f"https://www.twitch.tv/videos/{vod_id}"
    if duration_s:
        est = duration_s / DOWNLOAD_SPEED_X
        print(f"  → Descargando  (duración: {fmt_seconds(duration_s)}  |  estimado: ~{fmt_seconds(est)})")
    else:
        print(f"  → Descargando {twitch_url}...")

    cmd = [
        "yt-dlp", "--no-playlist",
        "--format", "audio_only/bestaudio",
        "--concurrent-fragments", "16",
        "--progress", "--newline",
        "-o", f"{dest}.%(ext)s",
        twitch_url,
    ]
    t0 = time.time()
    result = subprocess.run(cmd)
    elapsed = time.time() - t0
    if result.returncode != 0:
        print(f"  ✗ yt-dlp falló ({fmt_seconds(elapsed)})")
        return False
    print(f"  ✓ Descarga completada en {fmt_seconds(elapsed)}")
    return True


def find_audio_file(base: Path) -> Path | None:
    for ext in ("mp4", "ts", "aac", "mp3", "m4a", "opus", "webm", "ogg", "wav"):
        p = base.parent / f"{base.name}.{ext}"
        if p.exists():
            return p
    return None


def transcribe(audio_path: Path, model, model_name: str, duration_s, use_gpu: bool = False) -> list:
    speed_table = WHISPER_SPEED_GPU if use_gpu else WHISPER_SPEED_CPU
    speed_x = speed_table.get(model_name, 20.0 if use_gpu else 5.0)
    est_seconds = int(duration_s / speed_x) if duration_s else None

    if est_seconds:
        print(f"  → Transcribiendo '{model_name}'  (estimado: ~{fmt_seconds(est_seconds)})")
    else:
        print(f"  → Transcribiendo '{model_name}'...")

    t0 = time.time()
    segments_gen, info = model.transcribe(
        str(audio_path),
        language="en",
        beam_size=3,
        vad_filter=True,
        condition_on_previous_text=False,  # prevents hallucination loops on repetitive stream content
    )
    total_audio = info.duration if info.duration else (duration_s or 0)

    bar = tqdm(
        total=int(total_audio) if total_audio else None,
        unit="s", unit_scale=True, desc="    Whisper",
        bar_format="{l_bar}{bar}| {n:.0f}/{total:.0f}s [{elapsed}<{remaining}]" if total_audio else "{l_bar}{bar}| {elapsed}",
        dynamic_ncols=True,
    )

    entries, last_end = [], 0.0
    for seg in segments_gen:
        entries.append({
            "Text":    seg.text.strip(),
            "StartMs": int(seg.start * 1000),
            "EndMs":   int(seg.end   * 1000),
        })
        advance = seg.end - last_end
        if advance > 0:
            bar.update(int(advance))
            last_end = seg.end

    bar.close()
    elapsed = time.time() - t0
    print(f"  ✓ Completado en {fmt_seconds(elapsed)}  ({len(entries)} segmentos)")
    return entries


def save_cache(vod_id: str, entries: list) -> Path:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    path = CACHE_DIR / f"{vod_id}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(entries, f, indent=2, ensure_ascii=False)
    return path


def upload(vod_id: str, entries: list) -> dict:
    try:
        resp = requests.post(
            UPLOAD_URL,
            json={"video_id": vod_id, "entries": entries},
            timeout=30,
        )
        return {"code": resp.status_code, "body": resp.json()}
    except requests.exceptions.ConnectionError:
        return {"code": None, "body": {"error": "No se pudo conectar al servidor"}}
    except Exception as e:
        return {"code": None, "body": {"error": str(e)}}


def process_vod(vod_id: str, model, model_name: str, duration_s=None, use_gpu: bool = False) -> bool:
    print(f"\n{'─'*55}")
    title_str = f"[VOD {vod_id}]"
    if duration_s:
        title_str += f"  duración: {fmt_seconds(duration_s)}"
    print(title_str)

    vod_start = time.time()
    cached = CACHE_DIR / f"{vod_id}.json"

    if cached.exists():
        print(f"  → Cache encontrado, saltando descarga.")
        with open(cached, encoding="utf-8") as f:
            entries = json.load(f)
    else:
        tmp_dir = Path(tempfile.mkdtemp(prefix=f"vod_{vod_id}_"))
        audio_base = tmp_dir / vod_id
        try:
            if not download_audio(vod_id, audio_base, duration_s):
                return False
            audio_file = find_audio_file(audio_base)
            if not audio_file:
                print("  ✗ Archivo de audio no encontrado")
                return False
            entries = transcribe(audio_file, model, model_name, duration_s, use_gpu)
            if not entries:
                print("  ✗ Whisper no generó segmentos")
                return False
            path = save_cache(vod_id, entries)
            print(f"  → Guardado en cache: {path}")
        except Exception as e:
            print(f"  ✗ Error inesperado: {e}")
            return False
        finally:
            shutil.rmtree(tmp_dir, ignore_errors=True)

    print(f"  → Subiendo {len(entries)} segmentos...")
    res = upload(vod_id, entries)
    code, body = res["code"], res["body"]
    total = fmt_seconds(time.time() - vod_start)

    if code in (200, 201):
        verb = "Actualizado" if code == 200 else "Creado"
        print(f"  ✓ {verb} — {body.get('entries_saved', len(entries))} entradas  |  total: {total}")
        return True
    elif code == 404:
        print(f"  ✗ VOD no existe en la BD: {body.get('error', '')}")
        return False
    else:
        print(f"  ✗ Error [{code}]: {body}")
        return False


def _run_batch(vod_items: list, model, model_name: str, use_gpu: bool):
    """Procesa una lista de (vod_id, duration_s) con limpieza de memoria entre VODs."""
    speed_table = WHISPER_SPEED_GPU if use_gpu else WHISPER_SPEED_CPU
    speed_x = speed_table.get(model_name, 20.0 if use_gpu else 5.0)
    total_duration = sum(d or 0 for _, d in vod_items)
    est_download   = total_duration / DOWNLOAD_SPEED_X
    est_whisper    = total_duration / speed_x

    print(f"[INFO] {len(vod_items)} VODs  |  duración total: {fmt_seconds(total_duration)}")
    print(f"       Estimado descarga:  ~{fmt_seconds(est_download)}")
    print(f"       Estimado Whisper ({model_name} {'GPU' if use_gpu else 'CPU'}): ~{fmt_seconds(est_whisper)}")
    print(f"       Estimado total:    ~{fmt_seconds(est_download + est_whisper)}\n")

    global_start = time.time()
    success, failed = [], []

    for i, (vod_id, duration_s) in enumerate(vod_items, 1):
        print(f"[{i}/{len(vod_items)}]", end=" ")
        ok = process_vod(vod_id, model, model_name, duration_s, use_gpu)
        (success if ok else failed).append(vod_id)

        # Limpieza de memoria RAM + VRAM después de cada VOD
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    elapsed_total = time.time() - global_start
    print(f"\n{'═'*55}")
    print(f"Completado en {fmt_seconds(elapsed_total)}: {len(success)} exitosos  |  {len(failed)} fallidos")
    if failed:
        print(f"Fallidos: {failed}")


def process_list(vod_ids: list, model, model_name: str, use_gpu: bool = False):
    """Procesa una lista explícita de VOD IDs."""
    print(f"[INFO] Obteniendo info de {len(vod_ids)} VODs...")
    items = []
    for vod_id in vod_ids:
        info = get_video_info(vod_id)
        duration_s = info.get("length_seconds") if info else None
        items.append((vod_id, duration_s))
    _run_batch(items, model, model_name, use_gpu)


def process_all(model, model_name: str, use_gpu: bool = False):
    """Procesa todos los VODs del backend."""
    print("[INFO] Obteniendo lista de VODs desde la API...")
    vods = get_all_vods()
    if not vods:
        print("[ERROR] No hay VODs disponibles.")
        return
    items = [(v["id"], v.get("length_seconds")) for v in vods]
    _run_batch(items, model, model_name, use_gpu)


def process_pending(model, model_name: str, use_gpu: bool = False):
    """Fetch and process only VODs without transcripts for configured streamers."""
    print(f"[INFO] Buscando VODs sin transcripción para: {', '.join(STREAMER_LOGINS)}")
    vods = get_pending_vods(STREAMER_LOGINS)
    if not vods:
        print("✓ Todos los VODs ya tienen transcripción!")
        return 0
    items = [(v["id"], v.get("length_seconds")) for v in vods]
    _run_batch(items, model, model_name, use_gpu)
    return len(items)


print("✓ Funciones cargadas")

In [ ]:
# Celda 5 — Cargar modelo
# int8_float16: misma calidad que float16 pero ~50% menos VRAM en GPU
from faster_whisper import WhisperModel
compute_type = "int8_float16" if torch.cuda.is_available() else "int8"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = WhisperModel(MODEL_NAME, device=device, compute_type=compute_type)
print(f"✓ Modelo '{MODEL_NAME}' listo en {device} ({compute_type})")

In [ ]:
# Celda 6 — Procesar un VOD (VOD_ID de celda 3)
info = get_video_info(VOD_ID)
duration_s = info.get("length_seconds") if info else None
process_vod(VOD_ID, model, MODEL_NAME, duration_s, use_gpu=(device == "cuda"))

In [ ]:
# Celda 7 — Procesar lista de VODs (VOD_LIST de celda 3)
process_list(VOD_LIST, model, MODEL_NAME, use_gpu=(device == "cuda"))

In [ ]:
# Celda 8 — Procesar VODs pendientes (solo leonblack & shigity sin transcripción)
process_pending(model, MODEL_NAME, use_gpu=(device == "cuda"))

In [ ]:
# Celda 9 — Auto-loop: procesa pendientes hasta que no quede ninguno
# Ejecutar esta celda y dejar corriendo. Se detiene solo cuando todo está transcrito.
POLL_INTERVAL = 60  # segundos entre rondas

round_num = 0
while True:
    round_num += 1
    print(f"\n{'='*55}")
    print(f"RONDA {round_num}")
    print(f"{'='*55}")
    processed = process_pending(model, MODEL_NAME, use_gpu=(device == "cuda"))
    if processed == 0:
        print(f"\n✓ Todo listo — no quedan VODs pendientes.")
        break
    print(f"\nEsperando {POLL_INTERVAL}s antes de la siguiente ronda...")
    time.sleep(POLL_INTERVAL)